In [ ]:
from model.tiny_yolo import TinyYolo  
from settings import Settings  
import os  
  
# ============================================================  
# STEP 1: Update Settings for 2-class training  
# ============================================================  
# Make sure settings.py has:  
# - class_names = ["Barcode", "QR"]  
# - train_dataset = "data/train_barcode_qr.tf_record"  
# - val_dataset = "data/val_barcode_qr.tf_record"  
  
print("Current settings:")  
print(f"Classes: {Settings.class_names}")  
print(f"Train dataset: {Settings.train['train_dataset']}")  
print(f"Val dataset: {Settings.train['val_dataset']}") 

In [ ]:
# ============================================================  
# STEP 2: Create model with 2 classes and load checkpoint  
# ============================================================  
tinyolo = TinyYolo(training=True, classes=2)  
model = tinyolo._gen_model()  
  
# Load weights from your best checkpoint (epoch 99)  
checkpoint_path = "checkpoints/yolov3_train_99.weights.h5"  
print(f"\nLoading weights from: {checkpoint_path}")  
model.load_weights(checkpoint_path).expect_partial()  
print("✓ Weights loaded successfully") 

In [ ]:
# ============================================================  
# STEP 3: Configure separate logs directory  
# ============================================================  
# Override the logs directory to keep separate from original training  
import datetime  
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")  
new_log_dir = f"logs/finetune_2class_{timestamp}/"  
os.makedirs(new_log_dir, exist_ok=True)  
  
print(f"\nNew logs will be saved to: {new_log_dir}")  
print("Original logs remain in: logs/") 

In [ ]:
# ============================================================  
# STEP 4: Start fine-tuning  
# ============================================================  
# Temporarily override the log directory  
original_log_dir = Settings.train["logs"]  
Settings.train["logs"] = new_log_dir  
  
print("\n" + "="*60)  
print("Starting fine-tuning with 2 classes (Barcode + QR)")  
print("="*60)  
  
tinyolo.train(skip_transfer_learning=True) 
  
# Restore original log directory  
Settings.train["logs"] = original_log_dir  
  
print("\n" + "="*60)  
print("Fine-tuning complete!")  
print("="*60)  
print(f"View original training: tensorboard --logdir={original_log_dir}")  
print(f"View fine-tuning: tensorboard --logdir={new_log_dir}")  
print(f"Compare both: tensorboard --logdir=logs/")